# Module 20: Advanced Snowpark Patterns

**Snowpark ML Fundamentals — Day 2 Afternoon Session 1**

---

## Learning Objectives

Snowpark's built-in functions (`F.col`, `F.when`, `F.sum`) handle most transformations.
But production workloads need more:

| Pattern | When to Use |
|---------|------------|
| **Scalar UDFs** | Custom Python logic on each row (simple cases) |
| **Vectorized UDFs** | Batch Python logic via pandas (10-100x faster than scalar) |
| **Window functions** | Time-series features without collapsing rows |
| **Stored procedures** | Wrap complete workflows for scheduling |

## Estimated Time: 45 minutes

---

## 20.1 Setup (3 min)

In [ ]:
# --- Setup (given — just run this cell) ---
import sys
sys.path.insert(0, '../../../src')

import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

logger = logging.getLogger("module_20")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

from snowflake.snowpark import functions as F
from snowflake.snowpark import Window
from snowflake.snowpark.types import IntegerType, FloatType, StringType
from snowflake.snowpark.functions import udf, pandas_udf

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import (
    create_sample_orders_dataset, create_sample_customers_dataset,
)
from snowpark_fundamentals.data.loader import load_table

session = create_session(env_path='../../../.env')
logger.info("Session created.")

In [ ]:
# --- Schema isolation ---
STUDENT_NAME = "YOUR_NAME"  # <-- CHANGE THIS to your name (e.g., "MARIA")

session.sql("USE ROLE MLDS_ROLE_D").collect()
session.sql("USE DATABASE MLDS_D").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {STUDENT_NAME}_ONSITE_TRAINING").collect()
session.sql(f"USE SCHEMA {STUDENT_NAME}_ONSITE_TRAINING").collect()

# Generate datasets
orders_df = create_sample_orders_dataset(session, n_rows=5000)
customers_df = create_sample_customers_dataset(session, n_rows=500)

logger.info(f"Orders: {orders_df.count()} rows")
logger.info(f"Customers: {customers_df.count()} rows")
orders_df.show(5)

---

## 20.2 User-Defined Functions (UDFs) (15 min)

When Snowpark's built-in functions aren't enough, UDFs let you run custom Python
on every row. Two types:

| Type | How It Works | Speed | Use When |
|------|-------------|-------|----------|
| **Scalar UDF** | Processes one row at a time | Slower | Simple logic, non-numeric |
| **Vectorized UDF** | Processes batches via pandas Series | 10-100x faster | Numeric operations, always preferred |

**Rule:** Always use vectorized UDFs for numeric operations. Use scalar only when
the logic cannot be expressed on pandas Series.

In [ ]:
# --- Scalar UDF: categorize order amounts ---
# Processes one row at a time — simple but slower

def categorize_order(amount: float) -> str:
    """Categorize order by amount: HIGH / MEDIUM / LOW."""
    if amount > 500:
        return 'HIGH'
    elif amount > 100:
        return 'MEDIUM'
    else:
        return 'LOW'

categorize_udf = session.udf.register(
    func=categorize_order,
    name=f"{STUDENT_NAME}_CATEGORIZE_ORDER",
    return_type=StringType(),
    input_types=[FloatType()],
    is_permanent=False,
    replace=True,
)

# Apply it
result = orders_df.with_column('ORDER_CATEGORY', categorize_udf(F.col('ORDER_TOTAL')))
result.select('ORDER_ID', 'ORDER_TOTAL', 'ORDER_CATEGORY').show(5)

In [ ]:
# --- Vectorized UDF: recency score ---
# Processes batches via pandas Series — MUCH faster for numeric operations
import pandas as pd

@pandas_udf(
    name=f"{STUDENT_NAME}_RECENCY_SCORE",
    return_type=FloatType(),
    input_types=[IntegerType()],
    is_permanent=False,
    replace=True,
    packages=['pandas'],
    session=session,
)
def recency_score(days_series: pd.Series) -> pd.Series:
    """Compute recency score: recent orders score higher. Range: (0, 1]."""
    return 1.0 / (1.0 + days_series.clip(lower=0))

# Compute days since each order
orders_with_days = orders_df.with_column(
    'DAYS_SINCE_ORDER',
    F.datediff('day', F.col('ORDER_DATE'), F.current_date()),
)

# Apply vectorized UDF
result = orders_with_days.with_column(
    'RECENCY_SCORE',
    recency_score(F.col('DAYS_SINCE_ORDER')),
)
result.select('ORDER_ID', 'ORDER_DATE', 'DAYS_SINCE_ORDER', 'RECENCY_SCORE').show(5)
logger.info("Vectorized UDFs receive and return pandas Series.")
logger.info("Snowflake batches rows automatically — you get pandas speed at warehouse scale.")

### Your Turn: Write a Vectorized UDF

Create a **loyalty score** UDF that combines purchase signals:
- Formula: `loyalty_score = (item_count * order_total) / 1000.0`
- This measures both purchase frequency and monetary value in a single metric

In [ ]:
# TODO 1: Write a vectorized UDF that computes a "loyalty score"
# Formula: loyalty_score = (item_count * order_total) / 1000.0
# Hint: Follow the recency_score pattern — use @pandas_udf decorator
# Input: two pd.Series (item_count, order_total)
# Output: pd.Series

@pandas_udf(
    name=f"{STUDENT_NAME}_LOYALTY_SCORE",
    return_type=FloatType(),
    input_types=[IntegerType(), FloatType()],
    is_permanent=False,
    replace=True,
    packages=['pandas'],
    session=session,
)
def loyalty_score(item_count: pd.Series, order_total: pd.Series) -> pd.Series:
    ____

# Apply it
result = orders_df.with_column(
    'LOYALTY_SCORE',
    loyalty_score(F.col('ITEM_COUNT'), F.col('ORDER_TOTAL')),
)
result.select('ORDER_ID', 'ITEM_COUNT', 'ORDER_TOTAL', 'LOYALTY_SCORE').show(5)

In [ ]:
# --- Validation ---
assert 'LOYALTY_SCORE' in result.columns, "TODO 1: LOYALTY_SCORE column should exist"
sample = result.select('LOYALTY_SCORE').limit(10).to_pandas()
assert (sample['LOYALTY_SCORE'] >= 0).all(), "TODO 1: loyalty scores should be non-negative"
logger.info("TODO 1: Validation passed!")

---

## 20.3 Window Functions for Feature Engineering (12 min)

Window functions compute values across related rows **without collapsing** the
DataFrame. Unlike `group_by().agg()` (which reduces to one row per group),
window functions add a computed column to every row.

**Key components:**
- `Window.partition_by('COL')` — group rows (like GROUP BY, but keeps all rows)
- `.order_by('COL')` — define row ordering within each partition
- `.rows_between(start, end)` — define the window frame

### Four Essential Patterns

In [ ]:
# --- Pattern 1: Running total per customer ---
customer_window = Window.partition_by('CUSTOMER_ID').order_by('ORDER_DATE')

orders_running = orders_df.with_column(
    'CUMULATIVE_SPEND',
    F.sum('ORDER_TOTAL').over(customer_window),
)
logger.info("Pattern 1: Running total (cumulative spend per customer)")
orders_running.select(
    'CUSTOMER_ID', 'ORDER_DATE', 'ORDER_TOTAL', 'CUMULATIVE_SPEND'
).filter(F.col('CUSTOMER_ID') == 1).show(5)

In [ ]:
# --- Pattern 2: Rank customers by order amount ---
rank_window = Window.order_by(F.col('ORDER_TOTAL').desc())

orders_ranked = orders_df.with_column(
    'SPEND_RANK',
    F.rank().over(rank_window),
)
logger.info("Pattern 2: Rank (top 5 orders by amount)")
orders_ranked.select('ORDER_ID', 'ORDER_TOTAL', 'SPEND_RANK').show(5)

In [ ]:
# --- Pattern 3: Previous order (lag) ---
orders_with_lag = orders_df.with_column(
    'PREV_ORDER_TOTAL',
    F.lag('ORDER_TOTAL', 1).over(customer_window),
)
logger.info("Pattern 3: Lag (previous order amount per customer)")
orders_with_lag.select(
    'CUSTOMER_ID', 'ORDER_DATE', 'ORDER_TOTAL', 'PREV_ORDER_TOTAL'
).filter(F.col('CUSTOMER_ID') == 1).show(5)

In [ ]:
# --- Pattern 4: 3-order moving average ---
moving_window = (
    Window.partition_by('CUSTOMER_ID')
    .order_by('ORDER_DATE')
    .rows_between(-2, Window.CURRENT_ROW)
)

orders_with_ma = orders_df.with_column(
    'MOVING_AVG_3',
    F.avg('ORDER_TOTAL').over(moving_window),
)
logger.info("Pattern 4: 3-order moving average per customer")
orders_with_ma.select(
    'CUSTOMER_ID', 'ORDER_DATE', 'ORDER_TOTAL', 'MOVING_AVG_3'
).filter(F.col('CUSTOMER_ID') == 1).show(5)

### Your Turn: Create Window-Based Features

Create two features:
1. **PERCENTILE_RANK**: Each order's percentile position by amount (0.0 to 1.0)
2. **ORDER_CHANGE**: Difference between current order and previous order amount

In [ ]:
# TODO 2: Create two window-based features:
#   a) PERCENTILE_RANK: Percentile position of each order by ORDER_TOTAL (use F.percent_rank())
#      Hint: Window ordered by ORDER_TOTAL ascending, no partition (rank across all orders)
#   b) ORDER_CHANGE: Current order minus previous order per customer (use F.lag())
#      Hint: Use the customer_window defined above (partition by CUSTOMER_ID, order by ORDER_DATE)

# a) Percentile rank across all orders
pct_window = Window.order_by(F.col('ORDER_TOTAL'))
result = orders_df.with_column('PERCENTILE_RANK', ____)

# b) Order-over-order change per customer
customer_window = Window.partition_by('CUSTOMER_ID').order_by('ORDER_DATE')
result = result.with_column('ORDER_CHANGE', ____)

result.select(
    'CUSTOMER_ID', 'ORDER_DATE', 'ORDER_TOTAL',
    'PERCENTILE_RANK', 'ORDER_CHANGE',
).show(10)

In [ ]:
# --- Validation ---
assert 'PERCENTILE_RANK' in result.columns, "TODO 2a: PERCENTILE_RANK should exist"
assert 'ORDER_CHANGE' in result.columns, "TODO 2b: ORDER_CHANGE should exist"
sample = result.select('PERCENTILE_RANK').limit(20).to_pandas()
assert (sample['PERCENTILE_RANK'] >= 0).all() and (sample['PERCENTILE_RANK'] <= 1).all(), \
    "TODO 2a: PERCENTILE_RANK should be between 0 and 1"
logger.info("TODO 2: Validation passed!")

---

## 20.4 Stored Procedures for Workflow Orchestration (8 min)

Stored procedures wrap a complete workflow into a callable, schedulable unit.
This is how notebooks become production pipelines.

**When to use a stored procedure:**
- Scheduled retraining (via Snowflake Task)
- Batch scoring pipelines
- Data refresh and feature computation
- Champion/challenger automation (Module 18 pattern)

In [ ]:
# --- Scoring stored procedure ---
scoring_sp_sql = f"""
CREATE OR REPLACE PROCEDURE {STUDENT_NAME}_SCORE_ORDERS(
    INPUT_TABLE VARCHAR,
    OUTPUT_TABLE VARCHAR
)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
AS $$
def run(session, input_table: str, output_table: str) -> str:
    from snowflake.snowpark import functions as F

    # Read input
    df = session.table(input_table)

    # Apply business logic (categorize orders)
    scored = df.with_column(
        'ORDER_CATEGORY',
        F.when(F.col('ORDER_TOTAL') > 500, F.lit('HIGH'))
         .when(F.col('ORDER_TOTAL') > 100, F.lit('MEDIUM'))
         .otherwise(F.lit('LOW'))
    ).with_column(
        'SCORED_AT',
        F.current_timestamp()
    )

    # Write output
    scored.write.mode('overwrite').save_as_table(output_table)
    count = session.table(output_table).count()
    return f'Scored {{count}} orders into {{output_table}}'
$$;
"""

session.sql(scoring_sp_sql).collect()
logger.info("Stored procedure created.")
logger.info("Notice: all imports must be INSIDE the handler function.")
logger.info("The procedure runs in its own Python sandbox on the warehouse.")

In [ ]:
# TODO 3: Call the stored procedure to score orders
# Hint: Use session.sql("CALL procedure_name('arg1', 'arg2')")
# Arguments: input_table = 'SAMPLE_ORDERS', output_table = 'SCORED_ORDERS'

result = session.sql(f"""
    CALL {STUDENT_NAME}_SCORE_ORDERS(____, ____)
""").collect()

logger.info(f"Result: {result[0][0]}")
logger.info("\nScored orders sample:")
session.table('SCORED_ORDERS').show(5)

In [ ]:
# --- Validation ---
scored_count = session.table('SCORED_ORDERS').count()
assert scored_count > 0, "TODO 3: SCORED_ORDERS table should have rows"
scored_cols = session.table('SCORED_ORDERS').columns
assert 'ORDER_CATEGORY' in scored_cols, "TODO 3: ORDER_CATEGORY column should exist"
assert 'SCORED_AT' in scored_cols, "TODO 3: SCORED_AT column should exist"
logger.info(f"TODO 3: Validation passed! ({scored_count} orders scored)")

---

## 20.5 Performance Tips & Pandas Migration Pitfalls (5 min)

### Warehouse Sizing Quick Reference

| Size | Python Sandbox Memory | Best For |
|------|----------------------|----------|
| S | ~2 GB | Prototyping, small training (<50K rows) |
| M | ~8 GB | Production training (<500K rows) |
| L | ~32 GB | Large-scale training |
| Snowpark-Optimized M | ~256 GB | Memory-intensive workloads |

### Caching

Use `cache_result()` when a DataFrame is used in multiple downstream operations:
```python
# Without cache: preprocessing re-executes for EVERY model.fit()
# With cache: preprocessing runs ONCE, result is stored in a temp table
cached_df = expensive_preprocessing_df.cache_result()
```
See Module 15 for a detailed demo of caching impact.

### Top 5 Pandas → Snowpark Pitfalls

| # | Pandas Habit | Snowpark Reality | Fix |
|---|-------------|-----------------|-----|
| 1 | `df.apply(func)` | No row iteration | Use vectorized UDF |
| 2 | `df['col']` | Case sensitivity | `F.col('COL')` (uppercase) |
| 3 | Eager execution | Lazy evaluation | Call `.show()` / `.collect()` |
| 4 | `inplace=True` | Immutable DataFrames | Reassign: `df = df.with_column(...)` |
| 5 | `.to_pandas()` everywhere | Data transfer bottleneck | Keep data in Snowflake, use `.show()` |

**Full reference:** See the Pandas-to-Snowpark handout (`docs/pandas_to_snowpark_reference.md`)

---

## Key Takeaways

1. **Vectorized UDFs** for custom Python logic at scale — always prefer over scalar UDFs
2. **Window functions** for time-series features without group-by collapse —
   running totals, ranks, lags, moving averages
3. **Stored procedures** wrap workflows into callable, schedulable units
4. **Pandas → Snowpark** requires mindset shift: lazy execution, uppercase columns,
   no row iteration

**Next session:** Hands-on lab applying all three patterns →
Module 21: UDFs & Real-World Patterns Lab

In [ ]:
# --- Cleanup ---
try:
    session.sql("DROP TABLE IF EXISTS SCORED_ORDERS").collect()
    logger.info("Cleaned up.")
except Exception as e:
    logger.warning("Cleanup note: %s", e)

In [ ]:
session.close()